In [1]:
import torch
import warnings
warnings.filterwarnings("ignore")

from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CHROMA_DB_PATH = "../data/chroma_db"

print(f"Device       : {DEVICE}")
print(f"Torch version: {torch.__version__}")

Device       : cuda
Torch version: 2.6.0+cu124


In [2]:
# Load same embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": DEVICE},
    encode_kwargs={"normalize_embeddings": True}
)

# Load existing Chroma DB — no re-embedding!
vectorstore = Chroma(
    persist_directory=CHROMA_DB_PATH,
    embedding_function=embeddings
)

print(f"Chunks in DB : {vectorstore._collection.count()}")

# Set up retriever — fetch top 5 chunks
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

# Quick test
docs = retriever.invoke("What is deep learning?")
print(f"\nRetrieved {len(docs)} chunks for test query")
for d in docs:
    print(f"  → {d.metadata['title']}: {d.page_content[:100]}...")

/tmp/ipykernel_7527/2963807714.py:9: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


Chunks in DB : 211190

Retrieved 5 chunks for test query
  → Artificial intelligence: Deep learning 

Deep learning
uses several layers of neurons between the network's inputs and output...
  → Artificial intelligence: In reinforcement learning the agent is rewarded for good responses and punished for bad ones. The ag...
  → Artificial intelligence: Deep learning has drastically improved the performance of programs in many important subfields of ar...
  → Artificial intelligence: Deep learning began to dominate industry benchmarks in 2012 and was adopted throughout the field.
Fo...
  → Artificial intelligence: Modern deep learning techniques for NLP include word embedding (how often one word appears near anot...


In [3]:
llm = ChatOllama(
    model="llama3.1:8b",
    temperature=0,        # deterministic answers
    num_ctx=4096,         # context window
)

# Quick test
response = llm.invoke("Say hello in exactly one sentence.")
print("LLM response:", response.content)
print("LLM working  : ✓")

LLM response: Hello, how can I assist you today?
LLM working  : ✓


In [4]:
RAG_PROMPT = ChatPromptTemplate.from_template("""
You are a helpful assistant. Answer the question using ONLY the provided context.
If the context doesn't contain enough information, say "I don't have enough information to answer this."
Do not make up answers.

Context:
{context}

Question: {question}

Answer:""")

def format_docs(docs):
    """Combine retrieved chunks into single context string"""
    return "\n\n".join([
        f"[Source: {doc.metadata['title']}]\n{doc.page_content}"
        for doc in docs
    ])

# Test formatter
test_docs = retriever.invoke("What is the capital of France?")
print(format_docs(test_docs)[:500])

[Source: Assyria]
Capital cities

[Source: Brussels]
. Brussels is the de facto capital of the European Union, as it hosts a number of principal EU institutions, including its administrative-legislative, executive-political, and legislative branches (though the judicial branch is located in Luxembourg, and the European Parliament meets for a minority of the year in Strasbourg). Because of this, its name is sometimes used metonymically to describe the EU and its institutions

[Source: Belgium]
. 


In [5]:
# Build the chain using LCEL (LangChain Expression Language)
rag_chain = (
    {
        "context" : retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

print("RAG chain built successfully ✓")

RAG chain built successfully ✓


In [6]:
test_questions = [
    "What is machine learning?",
    "Who invented the telephone?",
    "What is the theory of relativity?",
    "What causes earthquakes?",
    "What is photosynthesis?"
]

print("=" * 60)
print("BASIC RAG — TEST RESULTS")
print("=" * 60)

for question in test_questions:
    print(f"\nQ: {question}")
    answer = rag_chain.invoke(question)
    print(f"A: {answer}")
    print("-" * 60)

BASIC RAG — TEST RESULTS

Q: What is machine learning?
A: Machine learning is the study of programs that can improve their performance on a given task automatically.
------------------------------------------------------------

Q: Who invented the telephone?
A: Alexander Graham Bell.
------------------------------------------------------------

Q: What is the theory of relativity?
A: I don't have enough information to answer this.

However, based on the provided context, I can provide some related answers:

* Special relativity is a part of the theory of relativity.
* General relativity is also a part of the theory of relativity.
* The theory of relativity includes both special and general relativity.
------------------------------------------------------------

Q: What causes earthquakes?
A: I don't have enough information to answer this. The provided context only mentions earthquakes in relation to their potential impact on climate change and disaster scenarios, but does not provide 

In [8]:
import os
# Helps prevent VRAM fragmentation
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()

In [9]:
from sentence_transformers import CrossEncoder
import torch

# Free up GPU memory first
torch.cuda.empty_cache()
print(f"GPU free before reranker: {(torch.cuda.memory_reserved() - torch.cuda.memory_allocated())/1e6:.1f} MB")

# Load reranker on CPU — it's fast enough for scoring 10 docs
reranker = CrossEncoder(
    "BAAI/bge-reranker-base",
    device="cpu"      # CPU only — saves GPU for LLM + embeddings
)

print("Reranker loaded on: CPU")
print("(CPU is fine for reranking — only scores 10 short texts per query)")

def reranked_retriever(query: str, k_retrieve: int = 10, k_final: int = 3):
    # Step 1 — retrieve broadly from vectorstore (GPU)
    docs = vectorstore.similarity_search(query, k=k_retrieve)
    
    # Step 2 — rerank on CPU (no VRAM needed)
    pairs = [[query, doc.page_content] for doc in docs]
    scores = reranker.predict(pairs)
    
    # Step 3 — sort and return top k_final
    scored_docs = sorted(
        zip(scores, docs),
        key=lambda x: x[0],
        reverse=True
    )
    
    top_docs  = [doc   for _, doc   in scored_docs[:k_final]]
    top_scores= [score for score, _ in scored_docs[:k_final]]
    
    print(f"Reranker scores: {[f'{s:.3f}' for s in top_scores]}")
    return top_docs

# Test
query    = "How does machine learning work?"
reranked = reranked_retriever(query)
print(f"\nTop {len(reranked)} reranked results:")
for i, doc in enumerate(reranked):
    print(f"\n{i+1}. [{doc.metadata['title']}]")
    print(doc.page_content[:200])

GPU free before reranker: 25.0 MB
Reranker loaded on: CPU
(CPU is fine for reranking — only scores 10 short texts per query)
Reranker scores: ['0.968', '0.953', '0.929']

Top 3 reranked results:

1. [Artificial intelligence]
There are several kinds of machine learning. Unsupervised learning analyzes a stream of data and finds patterns and makes predictions without any other guidance.
Supervised learning requires a human t

2. [Artificial intelligence]
Game theory describes rational behavior of multiple interacting agents, and is used in AI programs that make decisions that involve other agents.

Learning 
Machine learning is the study of programs t

3. [Computer programming]
Machine language


In [10]:
def rag_with_reranker(question: str) -> str:
    """Full RAG pipeline with reranking"""
    # Retrieve + rerank
    docs = reranked_retriever(question, k_retrieve=10, k_final=3)
    context = format_docs(docs)
    
    # Generate answer
    prompt = RAG_PROMPT.format_messages(
        context=context,
        question=question
    )
    response = llm.invoke(prompt)
    return response.content

# Compare basic vs reranked RAG
question = "What are the main causes of climate change?"

print("=" * 60)
print("BASIC RAG:")
print(rag_chain.invoke(question))

print("\n" + "=" * 60)
print("RERANKED RAG:")
print(rag_with_reranker(question))

BASIC RAG:
According to the provided context, the main causes of climate change mentioned are:

1. Human activities (specifically, burning fossil fuels and changes in land use)
2. Natural mechanisms (such as climate oscillations, changes in solar activity, volcanic activity)

Additionally, it is stated that the increase in greenhouse gases is "mostly responsible" for the observed warming over the past five decades.

RERANKED RAG:
Reranker scores: ['1.000', '0.999', '0.998']
I don't have enough information to answer this. The provided context only mentions some possible factors that could produce changes in global mean temperature and human activities that contribute to global warming, but it does not explicitly state the main causes of climate change.


In [11]:
import pandas as pd

# Quick eval on 5 CoQA questions
coqa_df = pd.read_csv("../data/processed/coqa_eval.csv")
sample_questions = coqa_df["question"].sample(5, random_state=42).tolist()
sample_answers   = coqa_df["answer"].sample(5, random_state=42).tolist()

print("=" * 60)
print("QUICK EVAL ON CoQA SAMPLES")
print("=" * 60)

for q, expected in zip(sample_questions, sample_answers):
    predicted = rag_chain.invoke(q)
    print(f"\nQ        : {q}")
    print(f"Expected : {expected}")
    print(f"Predicted: {predicted[:200]}")
    print("-" * 40)

print("\n✓ Basic RAG chain working!")
print("✓ Reranker working!")
print("\nReady for Phase 4 — Agentic RAG with LangGraph!")

QUICK EVAL ON CoQA SAMPLES

Q        : Where was it?
Expected : California.
Predicted: I don't have enough information to answer this.
----------------------------------------

Q        : in response to what?
Expected : the imposed conditions.
Predicted: I don't have enough information to answer this.
----------------------------------------

Q        : What store was the person going to?
Expected : pizza place
Predicted: I don't have enough information to answer this.
----------------------------------------

Q        : Do they own any studios?
Expected : yes
Predicted: I don't have enough information to answer this.
----------------------------------------

Q        : Who had been comforting her?
Expected : Kunelik and her son
Predicted: I don't have enough information to answer this.
----------------------------------------

✓ Basic RAG chain working!
✓ Reranker working!

Ready for Phase 4 — Agentic RAG with LangGraph!
